# bioRxiv Tools

![bioRxiv Tools](https://proto-bio.github.io/proto-assets/images/tool/biorxiv/hero.png)

This notebook demonstrates `run_biorxiv_details`, which looks up bioRxiv and medRxiv preprints by DOI. It returns every posted version with its date, the latest title, authors, category, license, and abstract, and the journal DOI once the preprint has been published. The bioRxiv API has no keyword search, so this notebook also shows how to find preprint DOIs with `run_europepmc_search`.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("biorxiv")
display_overview("biorxiv")
display_docs_section("biorxiv", "Background")

## Available tools

In [ ]:
display_available_tools("biorxiv")

### `run_biorxiv_details`

`run_biorxiv_details` takes one or more DOIs and returns one record per DOI. Here we look up the RFdiffusion preprint, which was revised once and later published in Nature.

In [ ]:
from proto_tools import (
    BiorxivDetailsConfig,
    BiorxivDetailsInput,
    run_biorxiv_details,
)

In [ ]:
# Display docs
display_api_reference("biorxiv", "input", "run_biorxiv_details")

# Look up the RFdiffusion preprint; a doi.org link is accepted
inputs = BiorxivDetailsInput(dois=["https://doi.org/10.1101/2022.12.09.519842"])

In [ ]:
# Display config docs
display_api_reference("biorxiv", "config", "run_biorxiv_details")

# Try bioRxiv, then medRxiv (the default) | see docs above for all fields
config = BiorxivDetailsConfig(server="auto")

In [ ]:
# Run the lookup
output = run_biorxiv_details(inputs, config)

In [ ]:
# Display output docs
display_api_reference("biorxiv", "output", "run_biorxiv_details")

# Inspect the record and its versions
preprint = output.preprints[0]
print(f"{preprint.title}\n{preprint.server}  {preprint.category}  license={preprint.license}")
print(f"Authors: {', '.join(preprint.authors[:5])}, ... ({len(preprint.authors)} total)")
print(f"Published as: {preprint.published_doi}")
for version in preprint.versions:
    print(f"  v{version.version}  {version.date}  {version.jats_xml_url}")

#### From a Keyword Search to Preprint Details

To find preprints by topic, search Europe PMC with `SRC:PPR` and pass the DOIs of the hits to `run_biorxiv_details`. Here we take the most recent bioRxiv hits for protein binder design and check their version history and publication status.

In [ ]:
from proto_tools import EuropePMCSearchConfig, EuropePMCSearchInput, run_europepmc_search

hits = run_europepmc_search(
    EuropePMCSearchInput(queries='TITLE:"binder design" AND SRC:PPR AND PUBLISHER:"bioRxiv"'),
    EuropePMCSearchConfig(max_results=3, include_abstracts=False),
).results[0].hits

details = run_biorxiv_details(BiorxivDetailsInput(dois=[hit.doi for hit in hits if hit.doi]))
for preprint in details.preprints:
    latest = preprint.versions[-1]
    print(f"{preprint.doi}  v{latest.version} ({latest.date})  published={preprint.published_doi}  {preprint.title[:60]}")

## Export Results

Records can be saved to JSON for downstream use. Preprint text and abstracts keep their authors' licenses, so this example writes to a temporary directory.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as output_dir:
    output.export("biorxiv_details", export_path=output_dir, file_format="json")
    print(f"Exported {sorted(path.name for path in Path(output_dir).iterdir())}")